# Lab 3 — Model Serving Microservice
Run only this notebook for Lab 3. Its prerequisite model preparation is included. Read START_HERE.md for Docker/UI tasks and submission instructions.

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, os, sys, subprocess, json, time, shutil
uploaded = files.upload()  # Choose Lab3_Model_Serving_Microservice.zip
zip_name = next(name for name in uploaded if name.endswith('.zip'))
base = Path('/content/cartvista-lab3')
base.mkdir(exist_ok=True)
with zipfile.ZipFile(zip_name) as z:
    # Reject path traversal; only extract the supplied source package.
    for name in z.namelist():
        if not (base / name).resolve().is_relative_to(base.resolve()):
            raise ValueError('Unsafe ZIP entry')
    z.extractall(base)
PROJECT = base / 'Lab3_Model_Serving_Microservice'
assert (PROJECT / 'requirements.txt').exists()
os.chdir(PROJECT)
print('Project:', PROJECT)


## Install isolated lab environment
The older MLflow/sklearn lab versions need Python 3.11/3.12. The separate environment avoids downgrading Colab's notebook packages. On a future newer hosted Python, this cell obtains a separate Python 3.11 using uv. Setup can take several minutes.


In [ ]:
AUTHOR = 'Sai YourFullName'  # CHANGE THIS BEFORE TRAINING
os.environ['LAB_AUTHOR'] = AUTHOR
VENV = Path('/content/cartvista-lab3-venv')
if sys.version_info[:2] in [(3,11),(3,12)]:
    subprocess.run([sys.executable, '-m', 'venv', str(VENV)], check=True)
else:
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'uv==0.8.22'], check=True)
    subprocess.run([sys.executable, '-m', 'uv', 'python', 'install', '3.11'], check=True)
    subprocess.run([sys.executable, '-m', 'uv', 'venv', '--seed', '--python', '3.11', str(VENV)], check=True)
PY = str(VENV / 'bin/python')
subprocess.run([PY, '-m', 'pip', 'install', '-r', str(PROJECT/'requirements.txt')], check=True)
def run(*args):
    p = subprocess.Popen([PY, '-u', *args], cwd=PROJECT, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    lines=[]
    for line in p.stdout:
        print(line, end=''); lines.append(line)
    status=p.wait()
    if status: raise subprocess.CalledProcessError(status, args, output=''.join(lines))
    return ''.join(lines)
run('-c', 'import mlflow,sklearn; print(mlflow.__version__,sklearn.__version__)')


## Prerequisite model preparation
This creates this lab’s own data, experiments and champion; do not move another machine’s SQLite registry here.

In [ ]:
run('run_lab1.py')

## Lab 3 Start the actual FastAPI service
The process listens inside your Colab VM. Requests from notebook cells use localhost. Keep one API process per port. The helper checks readiness and shows its log if startup fails.


In [ ]:
import requests
API = None
BASE='http://127.0.0.1:8000'
def stop_api():
    global API
    if API and API.poll() is None:
        API.terminate()
        try: API.wait(timeout=15)
        except subprocess.TimeoutExpired: API.kill(); API.wait()
def start_api(module, shadow=False, missing=False):
    global API
    stop_api()
    env=os.environ.copy();env['SHADOW_MODE']='1' if shadow else '0'
    if missing: env['MODEL_DIR']='/tmp/no-such-cartvista-model'
    else: env.pop('MODEL_DIR',None)
    API=subprocess.Popen([PY,'-m','uvicorn',module+':app','--host','0.0.0.0','--port','8000'],cwd=PROJECT,env=env,stdout=open('/content/cartvista-api.log','w'),stderr=subprocess.STDOUT)
    for _ in range(60):
        if API.poll() is not None: break
        try:
            health=requests.get(BASE+'/health',timeout=1).json()
            print(health)
            if not missing and not health['model_loaded']: raise RuntimeError(Path('/content/cartvista-api.log').read_text())
            return
        except requests.RequestException: time.sleep(1)
    raise RuntimeError(Path('/content/cartvista-api.log').read_text())
start_api('serve_churn')


In [ ]:
run('probe_service.py')
run('load_test.py')


### Alias at startup
Capture the old version, move the alias to a newly registered version, verify that the running service remains unchanged, then restart. The weights may be identical in the newly registered version; this exercise proves loading time, not an accuracy change.


In [ ]:
before=requests.get(BASE+'/model-info').json()
run('register_model.py')
after_alias=requests.get(BASE+'/model-info').json()
assert before==after_alias
start_api('serve_churn')
after_restart=requests.get(BASE+'/model-info').json()
assert after_restart['version']!=before['version']
proof={'before':before,'after_alias':after_alias,'after_restart':after_restart}
print(json.dumps(proof,indent=2))
(PROJECT/'evidence/lab3_alias.json').write_text(json.dumps(proof,indent=2))


### Missing model test
An invalid export path produces the same load-failure condition without renaming a live database. Health becomes degraded and valid prediction input receives 503.


In [ ]:
start_api('serve_churn',missing=True)
risk={'tenure_months':2,'monthly_spend':2500.,'orders_per_month':1,'support_tickets':6,'uses_discounts':0,'app_sessions':2}
r=requests.post(BASE+'/predict',json=risk);print(r.status_code,r.text);assert r.status_code==503
start_api('serve_churn')


## Download Lab 3 evidence
Save this executed notebook separately. Complete this lab’s report and any Docker/UI evidence described in START_HERE.md.

In [ ]:
stop_api()
stage=Path('/content/lab3-submission-evidence')
if stage.exists(): shutil.rmtree(stage)
stage.mkdir()
source=PROJECT/'evidence'
if source.exists():
    for f in source.rglob('*'):
        if not f.is_file(): continue
        # Include only this lab's results, plus relevant Colab-specific metrics.
        if f.name.startswith('lab3_') or False and (f.name.startswith('colab_') or 'pipeline' in f.parts or f.name=='gate_failures.log') or False and f.name.startswith('metrics_'):
            d=stage/'evidence'/f.relative_to(source);d.parent.mkdir(parents=True,exist_ok=True);shutil.copy2(f,d)
for name in ['churn_data.csv']:
    if (PROJECT/name).exists(): shutil.copy2(PROJECT/name,stage/name)
if (PROJECT/'model_export').exists(): shutil.copytree(PROJECT/'model_export',stage/'model_export')
archive=shutil.make_archive('/content/Lab3_My_Run_Evidence','zip',stage)
files.download(archive)
print('Finish the separate Lab 3 report and save this executed notebook.')
